## Bronze Layer: Ingestion Raw Data
raw data as ingested, no transformations
- sanitization: sanitize_numeric *dont use coercion to avoid loosing data.
- type casting: cast to appropriate types, but handle errors gracefully (e.g., log and keep original value if casting fails).

In [1]:
%load_ext autoreload
%autoreload 2

Import libraries

In [2]:
import sys
from pathlib import Path
project_root = Path.resolve(Path.cwd() / '../..')
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))

Load raw data

In [3]:
import pandas as pd
from notebooks.utils.preprocessing.data_loader import DataLoader
from notebooks.utils.preprocessing.basic_eda import BasicEDA

In [4]:
df = DataLoader.load(file_path='../../data/production_time.xlsx')

Loading data from ../../data/production_time.xlsx sheets...
Processing sheet: KROSNO I
Processing sheet: KROSNO II
Processing sheet: KROSNO III
Processing sheet: KROSNO IV
Processing sheet: KROSNO V
Skipping sheet: HARFY (not a production stand sheet)
Skipping sheet: SPAWANE (not a production stand sheet)
Skipping sheet: POZOSTAŁE (not a production stand sheet)
Skipping sheet: ZBIORCZE (not a production stand sheet)
Data loaded successfully with 4159 rows.


In [8]:
BasicEDA.basic_insights(df)

--------------------------------------------------
Basic EDA insights:
--------------------------------------------------
    Number of observations: 4159
    Duplicate rows: 71
    Shape: (4159, 11)
    Number of columns: 11

    Features:
        oczko/szczelina, Unnamed: 1, drut, *, szt., wymiar [mm], Unnamed: 6, szerokość [mm], zaciąg [mm], czas[min], stand

    Missing values:
        oczko/szczelina           0/4159     0.00%
        Unnamed: 1                0/4159     0.00%
        drut                      1/4159     0.02%
        *                      3609/4159    86.78%
        szt.                      2/4159     0.05%
        wymiar [mm]               0/4159     0.00%
        Unnamed: 6                0/4159     0.00%
        szerokość [mm]            0/4159     0.00%
        zaciąg [mm]               0/4159     0.00%
        czas[min]                 1/4159     0.02%
        stand                     0/4159     0.00%
--------------------------------------------------
End

Note:
    - `*` column is MNAR (missing not at random) - missing values are related to the opening shape (opening is square => width = length.

Data types and sample values

In [9]:
BasicEDA.show_samples(df)

--------------------------------------------------
Data types and sample values:
--------------------------------------------------
  oczko/szczelina    object   [2 1.5 4 2.5 3] ...
  Unnamed: 1         object   [2 70 4 87 3] ...
  drut               float64  [1.2 1.3 1.8 1.4 1.5] ...
  *                  object   [nan 'A' 'B' 'a' 'ST1'] ...
  szt.               object   [3 8 1 15 2] ...
  wymiar [mm]        int64    [1050 1020 1515 1390 1340] ...
  Unnamed: 6         object   [1805 2715 2160 2440 1825] ...
  szerokość [mm]     int64    [1050 1020 1515 1390 1340] ...
  zaciąg [mm]        float64  [ 6300. 21700.  3000.  3150. 28200.] ...
  czas[min]          float64  [390. 660. 240. 150. 750.] ...
  stand              object   ['KROSNO I' 'KROSNO II' 'KROSNO III' 'KROSNO IV' 'KROSNO V'] 



---
**Columns renaming**

In [10]:
column_names = [ 
    'op_w',             # oczko/szczelina       -> op_w         = opening width     [mm]
    'op_l',             # Unnamed: 1            -> op_l         = opening length    [mm]
    'wire_dia',         # drut                  -> wire_dia     = wire diameter     [mm]
    'op_align',         # *                     -> op_align     = opening alignment (A, B, nan)
    'qty',              # szt.                  -> qty          = product quantity  [pcs]
    'mesh_fl',          # wymiar [mm]           -> mesh_fl      = mesh weft width   [mm]
    'mesh_sp',          # Unnamed: 6            -> mesh_sp      = mesh warp length  [mm]
    'batch_width',      # szerokosc [mm]        -> batch_width  = batch width       [mm]
    'batch_length',     # zaciąg [mm]           -> batch_length = batch length      [mm]
    'process_time',     # czas[min]             -> process_time = production time of the batch [min]
    'stand_type'        # stand                 -> stand_type   = production stand type ('KROSNO I', 'KROSNO II', 'KROSNO III')
]

df.columns = column_names


---
**Initial Data Cleaning**
- handling missing values, converting data types

Columns to cast to float and int

In [11]:
columns_to_cast_float = ['op_w', 'op_l', 'wire_dia', 'mesh_fl', 'mesh_sp', 'batch_width', 'batch_length', 'process_time']
columns_to_cast_int = ['qty']
columns_to_cast_cat = ['op_align', 'stand_type']

numeric_cols = columns_to_cast_float + columns_to_cast_int

Initial non numeric values analysis

In [12]:
BasicEDA.analyse_numeric_columns(df, numeric_cols, sample_size=10)

Non-numeric values in "op_w"           found: 3    (values: 2,8N, 35X80, 15ST1)
Non-numeric values in "op_l"           found: 25   (values: 70N, 3N, 70N, 50OC, 30 ST1, 40ST1, 40ST1, 8ST1, 32ST1, 65ST1 ...)
Non-numeric values in "wire_dia"       found: 1    (values: nan)
Non-numeric values in "mesh_sp"        found: 17   (values: 10mb, 20mb, 30MB, 20mb, 20mb, 20MB, 760X2, 40 M2, 50M, 50M2 ...)
Non-numeric values in "process_time"   found: 1    (values: nan)
Non-numeric values in "qty"            found: 4    (values: ROLKA, 1 ROLKA, nan, nan)

Total unique indices with non-numeric values across all float columns: 45



In [13]:
print(df[df['op_align'].isna() & (df['op_w'] != df['op_l'])][['op_w', 'op_l', 'op_align']].head(10), '\n...')

    op_w op_l op_align
54     5  250      NaN
111    5    2      NaN
160    3   3N      NaN
207  2.8  70N      NaN
210  2.5   90      NaN
212  2.5  8.5      NaN
216  2.8   90      NaN
217  2.5   90      NaN
218  1.5   30      NaN
221  2.8   70      NaN 
...


Due to the presence of non-numeric values in columns that are clearly meant to be numeric and can be easily extracted as such, we will not use coercion to NaN for these columns as first step. Most of the non-numeric values are suffixed with string characters (e.g., 'mm', 'cm', 'm') that can be easily stripped to extract the numeric part. Only then we will apply coercion to NaN for any remaining non-numeric values. This way we can preserve the numeric information that is currently embedded in the non-numeric values, which would be lost if we directly coerced to NaN.

If sanitization introduces any wrong values, they will be handled in outliers detection, and basic input validation in silver layer.

In [14]:
from notebooks.utils.preprocessing.numeric_sanitizer import NumericSanitizer

# ===================== Sanitization + Type Casting ===================
df_sanitized = df.copy()

# 1. Sanitize numeric columns inplace
NumericSanitizer.sanitize_numeric_inplace(df_sanitized, numeric_cols)

# 2. Cast to appropriate types
schema = {
    **{col: 'float64' for col in columns_to_cast_float},
    **{col: 'int64' for col in columns_to_cast_int},
    **{col: 'category' for col in columns_to_cast_cat}
}
df_casted = df_sanitized.copy()

# Drop rows where numeric conversion failed
df_casted.dropna(subset=numeric_cols, inplace=True)

# Cast to target types
df_casted = df_casted.astype(schema, errors='raise')

# Reindex
df_casted.reset_index(drop=True, inplace=True)

print(df_casted.info())
print(f"Duplicated rows: {df_casted.duplicated().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4154 entries, 0 to 4153
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   op_w          4154 non-null   float64 
 1   op_l          4154 non-null   float64 
 2   wire_dia      4154 non-null   float64 
 3   op_align      549 non-null    category
 4   qty           4154 non-null   int64   
 5   mesh_fl       4154 non-null   float64 
 6   mesh_sp       4154 non-null   float64 
 7   batch_width   4154 non-null   float64 
 8   batch_length  4154 non-null   float64 
 9   process_time  4154 non-null   float64 
 10  stand_type    4154 non-null   category
dtypes: category(2), float64(8), int64(1)
memory usage: 300.7 KB
None
Duplicated rows: 71


In [16]:
import os; os.makedirs('../../data/dev/raw', exist_ok=True)
df_casted.to_parquet('../../data/dev/raw/production_time_bronze.parquet', index=False)
print("Data successfully ingested and saved to bronze layer as Parquet.")

Data successfully ingested and saved to bronze layer as Parquet.
